# Alerta Inteligente: comparação do ajuste fino (Entrega 3)

Grupo Sentinela do Cachoeira. Tarefa: classificar os relatos que os moradores mandam ao bot em
**confirmado**, **observacao** ou **nao_relacionado**.

Configurações comparadas, todas no mesmo conjunto de teste:

| Família | Configuração | O que é |
|---|---|---|
| BERTimbau | Base | `neuralmind/bert-base-portuguese-cased` sem nenhum treino |
| BERTimbau | Base com RAG | o BERTimbau base busca os 5 relatos de treino mais parecidos (base de exemplos) e a classe sai do voto deles |
| BERTimbau | Ajustado | o mesmo modelo depois do ajuste fino com os relatos de treino |
| BERTimbau | Ajustado com RAG | a resposta do ajustado combinada com o voto dos 5 exemplos mais parecidos |
| Gemini (referência) | Sem exemplos | `gemini-3.5-flash-lite` com o prompt v2 |
| Gemini (referência) | Com RAG | o mesmo prompt com os 5 exemplos mais parecidos recuperados da base |
| Referência simples | TF-IDF + regressão logística | o classificador que roda hoje no notebook do projeto |

Dois conjuntos de teste: o **teste** original (53 relatos de frases que o modelo nunca viu) e o **teste difícil**
(60 relatos novos com negação, gíria, fato passado, metáfora, vazamento doméstico e ordens escondidas).
Medimos acerto (acurácia e F1-macro), latência por relato (GPU e CPU) e custo por 1.000 relatos.

Antes de rodar: **Ambiente de execução > Alterar o tipo de ambiente de execução > GPU T4**.
Para o Gemini, a chave fica em **Secrets** (ícone de chave na barra lateral) com o nome `GEMINI_API_KEY`.
Depois: **Ambiente de execução > Executar tudo** e esperar (de 15 a 25 minutos, a maior parte é o Gemini).

In [ ]:
# 1. Baixa o repositório do projeto (público) e confere as bibliotecas
!rm -rf alerta-inteligente && git clone -q https://github.com/DiegoZarpok/alerta-inteligente.git
!pip -q install sentence-transformers
import torch, transformers, sklearn, pandas as pd, numpy as np
print("transformers", transformers.__version__, "| torch", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "nenhuma (vai ficar lento)")

In [ ]:
# 2. Dados: treino, validação, teste original e teste difícil
import random, time, json, re, copy, statistics
SEMENTE = 42
random.seed(SEMENTE); np.random.seed(SEMENTE); torch.manual_seed(SEMENTE)
PASTA = "alerta-inteligente/dados/finetuning/"
dados = pd.read_csv(PASTA + "relatos.csv")
dificil = pd.read_csv(PASTA + "relatos_dificeis.csv").reset_index(drop=True)
CLASSES = ["confirmado", "observacao", "nao_relacionado"]
rotulo = {c: i for i, c in enumerate(CLASSES)}
treino = dados[dados.divisao == "treino"].reset_index(drop=True)
validacao = dados[dados.divisao == "validacao"].reset_index(drop=True)
teste = dados[dados.divisao == "teste"].reset_index(drop=True)
CONJUNTOS = {"teste": teste, "teste_dificil": dificil}
print(pd.crosstab(pd.concat([dados, dificil]).classe, pd.concat([dados, dificil]).divisao))

In [ ]:
# 3. BERTimbau: tokenizador, modelo, probabilidades e vetores (para a busca do RAG)
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from torch.utils.data import DataLoader, TensorDataset
import torch.nn.functional as F

BASE = "neuralmind/bert-base-portuguese-cased"
GPU = "cuda" if torch.cuda.is_available() else "cpu"
tokenizador = AutoTokenizer.from_pretrained(BASE)

def novo_modelo():
    return AutoModelForSequenceClassification.from_pretrained(
        BASE, num_labels=3, id2label=dict(enumerate(CLASSES)), label2id=rotulo).to(GPU)

def onde(modelo):
    return next(modelo.parameters()).device

@torch.no_grad()
def probabilidades(modelo, textos, lote=32):
    modelo.eval(); saida = []
    for i in range(0, len(textos), lote):
        enc = tokenizador(list(textos[i:i + lote]), truncation=True, padding=True, max_length=96,
                          return_tensors="pt").to(onde(modelo))
        saida.append(F.softmax(modelo(**enc).logits, dim=-1).cpu())
    return torch.cat(saida).numpy()

@torch.no_grad()
def vetores(modelo, textos, lote=32):
    """Média dos vetores da última camada do codificador (BERT), normalizada."""
    codificador = modelo.base_model
    codificador.eval(); saida = []
    for i in range(0, len(textos), lote):
        enc = tokenizador(list(textos[i:i + lote]), truncation=True, padding=True, max_length=96,
                          return_tensors="pt").to(onde(modelo))
        h = codificador(**enc).last_hidden_state
        m = enc["attention_mask"].unsqueeze(-1).float()
        saida.append(F.normalize((h * m).sum(1) / m.sum(1), dim=-1).cpu())
    return torch.cat(saida).numpy()

K = 5  # exemplos recuperados pelo RAG

def votos(modelo, matriz_base, textos):
    """RAG da classificação: busca os K relatos de treino mais parecidos e soma a similaridade por classe."""
    q = vetores(modelo, textos)
    sims = q @ matriz_base.T
    topo = np.argsort(-sims, axis=1)[:, :K]
    dist = np.zeros((len(textos), len(CLASSES)))
    for i in range(len(textos)):
        for j in topo[i]:
            dist[i, rotulo[treino.classe[j]]] += max(float(sims[i, j]), 0.0)
    dist = dist / np.clip(dist.sum(1, keepdims=True), 1e-9, None)
    return dist, topo

In [ ]:
# 4. Medição: acerto, latência e custo de cada configuração nos dois conjuntos
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix
resultados, erros_dificil, matrizes = [], {}, {}

def avaliar(nome, familia, prever, custo_1000_usd=0.0):
    """prever(textos) devolve a lista de classes (e pode guardar custo em CUSTO_ATUAL)."""
    for conjunto, df in CONJUNTOS.items():
        inicio = time.perf_counter()
        previsto = prever(list(df.texto))
        tempo = time.perf_counter() - inicio
        linha = {"configuracao": nome, "familia": familia, "conjunto": conjunto, "n": len(df),
                 "acuracia": round(accuracy_score(df.classe, previsto), 3),
                 "f1_macro": round(f1_score(df.classe, previsto, average="macro", labels=CLASSES), 3),
                 "tempo_total_s": round(tempo, 2), "custo_1000_usd": custo_1000_usd}
        resultados.append(linha)
        matrizes[f"{nome} | {conjunto}"] = confusion_matrix(df.classe, previsto, labels=CLASSES).tolist()
        if conjunto == "teste_dificil":
            erros_dificil[nome] = [{"texto": t, "certo": c, "previsto": p}
                                   for t, c, p in zip(df.texto, df.classe, previsto) if c != p]
        print(f"{nome:<32} {conjunto:<14} acurácia {linha['acuracia']:.3f}  F1 {linha['f1_macro']:.3f}")

def latencia_ms(prever, textos):
    """Tempo de UM relato por vez (como o bot recebe), mediana em milissegundos."""
    tempos = []
    for t in textos:
        inicio = time.perf_counter(); prever([t]); tempos.append((time.perf_counter() - inicio) * 1000)
    return round(statistics.median(tempos), 1)

AMOSTRA_LATENCIA = list(teste.texto[:30])
latencias = {}

In [ ]:
# 5. BERTimbau BASE, sem ajuste (a camada de classificação ainda não foi treinada)
torch.manual_seed(SEMENTE)
base = novo_modelo()
prever_base = lambda textos: [CLASSES[i] for i in probabilidades(base, textos).argmax(1)]
avaliar("BERTimbau base", "BERTimbau", prever_base)

# 6. BERTimbau BASE COM RAG: os vetores do próprio BERTimbau base buscam os 5 exemplos de treino mais parecidos
matriz_base = vetores(base, list(treino.texto))
prever_base_rag = lambda textos: [CLASSES[i] for i in votos(base, matriz_base, textos)[0].argmax(1)]
avaliar("BERTimbau base com RAG", "BERTimbau", prever_base_rag)
latencias["BERTimbau base"] = {"gpu": latencia_ms(prever_base, AMOSTRA_LATENCIA)}
latencias["BERTimbau base com RAG"] = {"gpu": latencia_ms(prever_base_rag, AMOSTRA_LATENCIA)}

In [ ]:
# 7. Ajuste fino (mesma receita da Entrega 2)
def carregar(df, embaralhar):
    enc = tokenizador(list(df.texto), truncation=True, padding="max_length", max_length=96, return_tensors="pt")
    y = torch.tensor([rotulo[c] for c in df.classe])
    return DataLoader(TensorDataset(enc["input_ids"], enc["attention_mask"], y), batch_size=16, shuffle=embaralhar)

dl_treino = carregar(treino, True)
torch.manual_seed(SEMENTE)
modelo = novo_modelo()
EPOCAS, TAXA = 6, 3e-5
otimizador = torch.optim.AdamW(modelo.parameters(), lr=TAXA, weight_decay=0.01)
passos = EPOCAS * len(dl_treino)
agenda = get_linear_schedule_with_warmup(otimizador, int(0.1 * passos), passos)
melhor_f1, melhor_estado, historico = -1, None, []
inicio = time.time()
for epoca in range(1, EPOCAS + 1):
    modelo.train(); perda_total = 0
    for ids, mascara, y in dl_treino:
        otimizador.zero_grad()
        saida = modelo(input_ids=ids.to(GPU), attention_mask=mascara.to(GPU), labels=y.to(GPU))
        saida.loss.backward()
        torch.nn.utils.clip_grad_norm_(modelo.parameters(), 1.0)
        otimizador.step(); agenda.step()
        perda_total += saida.loss.item()
    prev_val = [CLASSES[i] for i in probabilidades(modelo, list(validacao.texto)).argmax(1)]
    f1_val = f1_score(validacao.classe, prev_val, average="macro")
    historico.append({"epoca": epoca, "perda_treino": round(perda_total / len(dl_treino), 4), "f1_validacao": round(f1_val, 3)})
    print(historico[-1])
    if f1_val > melhor_f1:
        melhor_f1, melhor_estado = f1_val, {k: v.detach().cpu().clone() for k, v in modelo.state_dict().items()}
tempo_treino = round(time.time() - inicio, 1)
modelo.load_state_dict(melhor_estado)
print(f"Treino em {tempo_treino} s na GPU; melhor F1 na validação: {melhor_f1:.3f}")

In [ ]:
# 8. AJUSTADO e AJUSTADO COM RAG (resposta do modelo + voto dos 5 exemplos mais parecidos, meio a meio)
prever_ajustado = lambda textos: [CLASSES[i] for i in probabilidades(modelo, textos).argmax(1)]
avaliar("BERTimbau ajustado", "BERTimbau", prever_ajustado)
matriz_ajustado = vetores(modelo, list(treino.texto))
def prever_ajustado_rag(textos):
    p = probabilidades(modelo, textos)
    dist, _ = votos(modelo, matriz_ajustado, textos)
    return [CLASSES[i] for i in (0.5 * p + 0.5 * dist).argmax(1)]
avaliar("BERTimbau ajustado com RAG", "BERTimbau", prever_ajustado_rag)
latencias["BERTimbau ajustado"] = {"gpu": latencia_ms(prever_ajustado, AMOSTRA_LATENCIA)}
latencias["BERTimbau ajustado com RAG"] = {"gpu": latencia_ms(prever_ajustado_rag, AMOSTRA_LATENCIA)}

In [ ]:
# 9. Latência em CPU (o notebook do projeto não tem GPU): as mesmas quatro configurações, um relato por vez
base_gpu, modelo_gpu = base, modelo
base, modelo = copy.deepcopy(base_gpu).to("cpu"), copy.deepcopy(modelo_gpu).to("cpu")
for nome, funcao in [("BERTimbau base", prever_base), ("BERTimbau base com RAG", prever_base_rag),
                     ("BERTimbau ajustado", prever_ajustado), ("BERTimbau ajustado com RAG", prever_ajustado_rag)]:
    latencias[nome]["cpu"] = latencia_ms(funcao, AMOSTRA_LATENCIA[:15])
base, modelo = base_gpu, modelo_gpu
print(json.dumps(latencias, indent=1))

In [ ]:
# 10. Referência simples: TF-IDF + regressão logística (o que roda hoje no notebook do projeto)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
tfidf = make_pipeline(TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True),
                      LogisticRegression(max_iter=2000, class_weight="balanced"))
tfidf.fit(treino.texto, treino.classe)
prever_tfidf = lambda textos: list(tfidf.predict(textos))
avaliar("TF-IDF + regressão logística", "Referência", prever_tfidf)
latencias["TF-IDF + regressão logística"] = {"cpu": latencia_ms(prever_tfidf, AMOSTRA_LATENCIA)}

In [ ]:
# 11. Gemini sem exemplos e Gemini com RAG (prompt v2 da pasta prompts/)
import requests
from sentence_transformers import SentenceTransformer
from google.colab import userdata
MODELO_GEMINI = "gemini-3.5-flash-lite"
PRECO_ENTRADA, PRECO_SAIDA = 0.30, 2.50  # US$ por 1 milhão de tokens (o mesmo de src/ia/llm.py)
texto_prompt = open("alerta-inteligente/prompts/classificar_relato_v2.md", encoding="utf-8").read()
SISTEMA = re.search(r"## Sistema\n(.*?)\n## Usuario\n", texto_prompt, re.S).group(1).strip()
USUARIO = re.search(r"## Usuario\n(.*?)\n## Usuario com exemplos", texto_prompt, re.S).group(1).strip()
USUARIO_RAG = re.search(r"## Usuario com exemplos\n(.*)$", texto_prompt, re.S).group(1).strip()

# a busca do RAG usa o mesmo modelo de vetores do projeto (MiniLM, 384 dimensões)
buscador = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
matriz_exemplos = buscador.encode(list(treino.texto), normalize_embeddings=True)
def exemplos(texto):
    q = buscador.encode([texto], normalize_embeddings=True)[0]
    topo = np.argsort(-(matriz_exemplos @ q))[:K]
    return "\n".join(f'- "{treino.texto[j]}" -> {treino.classe[j]}' for j in topo)

try:
    CHAVE = userdata.get("GEMINI_API_KEY")
except Exception:
    CHAVE = None
URL = f"https://generativelanguage.googleapis.com/v1beta/models/{MODELO_GEMINI}:generateContent"
gasto = {}

def gemini(usuario):
    corpo = {"systemInstruction": {"parts": [{"text": SISTEMA}]},
             "contents": [{"role": "user", "parts": [{"text": usuario}]}],
             "generationConfig": {"responseMimeType": "application/json", "maxOutputTokens": 30,
                                  "thinkingConfig": {"thinkingLevel": "minimal"}}}
    for tentativa in range(6):
        r = requests.post(URL, headers={"x-goog-api-key": CHAVE}, json=corpo, timeout=60)
        if r.status_code == 200:
            break
        if r.status_code == 400 and "thinking" in r.text.lower():
            corpo["generationConfig"].pop("thinkingConfig", None); continue
        time.sleep(15 * (tentativa + 1))  # 429 (cota por minuto) ou 503: espera e tenta de novo
    if r.status_code != 200:
        return "nao_relacionado", 0, 0, False
    dados_r = r.json(); uso = dados_r.get("usageMetadata", {})
    partes = dados_r["candidates"][0].get("content", {}).get("parts", [])
    texto = "".join(p.get("text", "") for p in partes if not p.get("thought"))
    m = re.search(r'"classe"\s*:\s*"(\w+)"', texto)
    classe = m.group(1) if m and m.group(1) in CLASSES else "nao_relacionado"
    return classe, uso.get("promptTokenCount", 0), uso.get("candidatesTokenCount", 0) + uso.get("thoughtsTokenCount", 0), True

def prever_gemini(com_rag):
    def prever(textos):
        saida = []
        for t in textos:
            usuario = USUARIO_RAG.replace("{exemplos}", exemplos(t)).replace("{texto}", t) if com_rag \
                else USUARIO.replace("{texto}", t)
            inicio = time.perf_counter()
            classe, ent, sai, ok = gemini(usuario)
            g = gasto.setdefault(com_rag, {"tempos": [], "entrada": 0, "saida": 0, "falhas": 0, "chamadas": 0})
            g["tempos"].append(time.perf_counter() - inicio); g["entrada"] += ent; g["saida"] += sai
            g["falhas"] += 0 if ok else 1; g["chamadas"] += 1
            saida.append(classe)
            time.sleep(1)
        return saida
    return prever

if not CHAVE:
    print("Sem GEMINI_API_KEY nos Secrets: pulando o Gemini.")
else:
    for com_rag, nome in [(False, "Gemini sem exemplos"), (True, "Gemini com RAG")]:
        avaliar(nome, "Gemini (referência)", prever_gemini(com_rag))
        g = gasto[com_rag]
        custo = (g["entrada"] * PRECO_ENTRADA + g["saida"] * PRECO_SAIDA) / 1e6 / g["chamadas"] * 1000
        for linha in resultados:
            if linha["configuracao"] == nome:
                linha["custo_1000_usd"] = round(custo, 3)
        latencias[nome] = {"api": round(statistics.median(g["tempos"]) * 1000, 1)}
        print(f"{nome}: {g['chamadas']} chamadas, {g['falhas']} falhas, custo por 1.000 relatos US$ {custo:.3f}, "
              f"mediana {latencias[nome]['api']} ms")

In [ ]:
# 12. Tabela final, erros no teste difícil e arquivos para o relatório
tabela = pd.DataFrame(resultados)
tabela["latencia_gpu_ms"] = tabela.configuracao.map(lambda n: latencias.get(n, {}).get("gpu"))
tabela["latencia_cpu_ms"] = tabela.configuracao.map(lambda n: latencias.get(n, {}).get("cpu"))
tabela["latencia_api_ms"] = tabela.configuracao.map(lambda n: latencias.get(n, {}).get("api"))
pd.set_option("display.width", 200)
print(tabela.pivot_table(index="configuracao", columns="conjunto", values=["acuracia", "f1_macro"], sort=False))
print()
print(tabela.drop_duplicates("configuracao")[["configuracao", "latencia_gpu_ms", "latencia_cpu_ms", "latencia_api_ms", "custo_1000_usd"]].to_string(index=False))
for nome, erros in erros_dificil.items():
    print(f"\n{nome}: {len(erros)} erros no teste difícil")
    for e in erros[:6]:
        print(f"   certo {e['certo']:<15} previsto {e['previsto']:<15} {e['texto'][:80]}")

saida = {"data": time.strftime("%Y-%m-%d %H:%M"), "base": BASE, "epocas": EPOCAS, "taxa_aprendizado": TAXA,
         "tempo_treino_s": tempo_treino, "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
         "historico": historico, "k_rag": K, "modelo_gemini": MODELO_GEMINI,
         "tamanhos": {"treino": len(treino), "validacao": len(validacao), "teste": len(teste), "teste_dificil": len(dificil)},
         "resultados": resultados, "latencias_ms": latencias, "matrizes": matrizes, "classes": CLASSES,
         "erros_teste_dificil": erros_dificil,
         "gemini": {("com_rag" if k else "sem_exemplos"): {kk: vv for kk, vv in v.items() if kk != "tempos"}
                    for k, v in gasto.items()}}
json.dump(saida, open("finetuning_comparacao.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
from google.colab import files
files.download("finetuning_comparacao.json")